In [ ]:
import nfl_data_py as nfl
import pandas as pd

# ── 1. Load play-by-play for 2020–2024 ──────────────────────────────────────
seasons = [2020, 2021, 2022, 2023, 2024]

pbp = nfl.import_pbp_data(seasons)

# Keep only pass plays with a coverage defender tagged
passes = pbp[
    (pbp['pass'] == 1) &
    (pbp['season_type'] == 'REG') &
    (pbp['pass_defense_1_player_id'].notna())
].copy()

# ── 2. Build per-play CB rows ────────────────────────────────────────────────
# Each play can have up to 2 defenders — treat each as a separate row
def extract_defender(df, slot):
    pid_col = f'pass_defense_{slot}_player_id'
    pname_col = f'pass_defense_{slot}_player_name'
    return df[[
        'season', pid_col, pname_col,
        'complete_pass', 'incomplete_pass', 'interception',
        'pass_touchdown', 'yards_gained'
    ]].rename(columns={pid_col: 'player_id', pname_col: 'player_name'})

defenders = pd.concat([
    extract_defender(passes, 1),
    extract_defender(passes[passes['pass_defense_2_player_id'].notna()], 2)
])

# ── 3. Aggregate to season-level CB stats ───────────────────────────────────
cb_stats = defenders.groupby(['season', 'player_id', 'player_name']).agg(
    targets       = ('complete_pass',    'count'),      # every row = 1 target
    comp_allowed  = ('complete_pass',    'sum'),
    incompletions = ('incomplete_pass',  'sum'),
    interceptions = ('interception',     'sum'),
    td_allowed    = ('pass_touchdown',   'sum'),
    yards_allowed = ('yards_gained',     'sum'),
).reset_index()

# Derived rates
cb_stats['comp_pct_allowed'] = cb_stats['comp_allowed'] / cb_stats['targets']
cb_stats['int_rate']         = cb_stats['interceptions'] / cb_stats['targets']
cb_stats['td_rate_allowed']  = cb_stats['td_allowed'] / cb_stats['targets']
cb_stats['yds_per_target']   = cb_stats['yards_allowed'] / cb_stats['targets']

# ── 4. Load snap counts and filter to CBs with 2500+ snaps ──────────────────
snaps = nfl.import_snap_counts(seasons)

cb_snaps = snaps[snaps['position'] == 'CB'].groupby(
    ['season', 'player_id']
).agg(total_snaps=('defense_snaps', 'sum')).reset_index()

# Filter: 2500+ snaps across all 5 seasons
snap_totals = cb_snaps.groupby('player_id')['total_snaps'].sum().reset_index()
qualified   = snap_totals[snap_totals['total_snaps'] >= 2500]['player_id']

# ── 5. Merge and export ──────────────────────────────────────────────────────
final = (
    cb_stats
    .merge(cb_snaps, on=['season', 'player_id'], how='left')
    .merge(snap_totals.rename(columns={'total_snaps': 'career_snaps'}),
           on='player_id', how='left')
    .query('player_id in @qualified')
)

# Add target rate vs league average (per 100 snaps)
league_avg_target_rate = final['targets'].sum() / final['total_snaps'].sum() * 100
final['target_rate']         = final['targets'] / final['total_snaps'] * 100
final['target_rate_vs_avg']  = final['target_rate'] - league_avg_target_rate

final.to_csv('cb_coverage_data_2020_2024.csv', index=False)
print(f"Dataset: {final['player_id'].nunique()} CBs, {len(final)} season-rows")

ModuleNotFoundError: No module named 'appdirs'

: 

In [5]:
import sys
!{sys.executable} -m pip install appdirs

'c:\Users\adars\OneDrive\Desktop\Pigskin' is not recognized as an internal or external command,
operable program or batch file.
